<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.1-tokenization-and-embeddings/notebooks/exercises-2.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2.1: Tokenization & Embeddings — Practice Exercises

**Netsetos GenAI Course v2.0** | Module 2 | 6 exercises: BPE exploration, token tax, special tokens, embeddings comparison, similarity search, multilingual tokenization.

---

In [ ]:
!pip install transformers tiktoken sentence-transformers -q


---
## Exercise 1 (Easy): BPE Tokenizer Explorer
Tokenize the same text with GPT-2, GPT-4, and BERT tokenizers. Compare token counts.

In [ ]:
import tiktoken
from transformers import AutoTokenizer

text = 'Hyderabad is a beautiful city in Telangana, India. The biryani here is amazing!'

# GPT-2 tokenizer
gpt2_enc = tiktoken.encoding_for_model('gpt-2')
gpt2_tokens = gpt2_enc.encode(text)
print(f'GPT-2:  {len(gpt2_tokens)} tokens → {gpt2_enc.decode_tokens_bytes(gpt2_tokens)}')

# GPT-4o tokenizer
gpt4_enc = tiktoken.encoding_for_model('gpt-4o')
gpt4_tokens = gpt4_enc.encode(text)
print(f'GPT-4o: {len(gpt4_tokens)} tokens → {[gpt4_enc.decode([t]) for t in gpt4_tokens]}')

# BERT tokenizer (WordPiece)
bert_tok = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_tokens = bert_tok.tokenize(text.lower())
print(f'BERT:   {len(bert_tokens)} tokens → {bert_tokens}')

print(f'\nGPT-4o is more efficient — fewer tokens for the same text!')


---
## Exercise 2 (Easy): The 'Strawberry' Problem + Token Tax
How many r's in 'strawberry'? Why do LLMs get this wrong? Explore the multilingual token tax.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o')

# The strawberry problem
word = 'strawberry'
tokens = enc.encode(word)
print(f"'{word}' → {len(tokens)} tokens: {[enc.decode([t]) for t in tokens]}")
print(f'The model sees tokens, not characters. It cannot count r\'s!')

# Multilingual token tax
texts = {
    'English': 'The weather is nice today',
    'Hindi': 'आज मौसम बहुत अच्छा है',
    'Telugu': 'ఈ రోజు వాతావరణం చాలా బాగుంది',
    'Arabic': 'الطقس جميل اليوم',
    'Japanese': '今日はいい天気ですね',
}

print(f'\nMULTILINGUAL TOKEN TAX:')
print(f'{"Language":<12} {"Tokens":>7} {"Tax vs English":>15}')
print('-' * 40)
en_count = len(enc.encode(texts['English']))
for lang, text in texts.items():
    count = len(enc.encode(text))
    tax = f'{count/en_count:.1f}x'
    print(f'{lang:<12} {count:>7} {tax:>15}')
print('\nHindi/Telugu cost 2-3x more tokens → 2-3x higher API cost!')


---
## Exercise 3 (Medium): Special Tokens & Chat Templates
Explore [CLS], [SEP], [MASK], <|endoftext|>, and chat template formatting.

In [ ]:
from transformers import AutoTokenizer

# BERT special tokens
bert = AutoTokenizer.from_pretrained('bert-base-uncased')
encoded = bert('Hello world', 'How are you?', return_tensors='pt')
tokens = bert.convert_ids_to_tokens(encoded['input_ids'][0])
print('BERT special tokens:')
print(f'  {tokens}')
print(f'  [CLS]=sentence start, [SEP]=separator, padding\n')

# GPT-2 special tokens
gpt2 = AutoTokenizer.from_pretrained('gpt2')
print(f'GPT-2 EOS token: "{gpt2.eos_token}" (id={gpt2.eos_token_id})')

# Chat template (LLaMA-style)
print('\nChat template (conceptual):')
print('  <|system|>You are a helpful assistant.<|end|>')
print('  <|user|>What is 2+2?<|end|>')
print('  <|assistant|>4<|end|>')
print('\nUnder the hood: concatenated into ONE token sequence for next-token prediction!')


---
## Exercise 4 (Medium): Word2Vec vs BERT Embeddings
Show that BERT gives 'bank' different vectors in different contexts (contextual embeddings).

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained('bert-base-uncased')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

sentences = [
    'I deposited money at the bank',        # financial
    'We sat by the river bank',              # geographical
    'The bank approved my loan application', # financial
]

bank_vectors = []
for s in sentences:
    inputs = tokenizer(s, return_tensors='pt')
    tokens = tokenizer.tokenize(s)
    bank_idx = tokens.index('bank') + 1  # +1 for [CLS]
    with torch.no_grad():
        out = model(**inputs)
        vec = out.last_hidden_state[0, bank_idx].numpy()
        bank_vectors.append(vec)

def cosine(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print('CONTEXTUAL EMBEDDINGS — same word, different vectors:')
print(f'  financial vs river:     {cosine(bank_vectors[0], bank_vectors[1]):.3f}')
print(f'  financial vs financial: {cosine(bank_vectors[0], bank_vectors[2]):.3f}')
print(f'\nSame-meaning "bank" has HIGH similarity (~0.8+)')
print(f'Different-meaning "bank" has LOWER similarity (~0.6)')
print(f'Word2Vec would give ONE vector for all — this is the breakthrough!')


---
## Exercise 5 (Hard): Semantic Similarity Search
Build a mini search engine using sentence embeddings + cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Document corpus
docs = [
    'Python is a popular programming language for AI',
    'The Taj Mahal is located in Agra, India',
    'Machine learning uses data to make predictions',
    'Hyderabad is known for its delicious biryani',
    'Neural networks are inspired by the human brain',
    'Cricket is the most popular sport in India',
]

doc_embeddings = model.encode(docs)

# Search
queries = ['Tell me about AI', 'Indian food', 'sports']
for query in queries:
    q_emb = model.encode([query])
    similarities = np.dot(doc_embeddings, q_emb.T).flatten()
    best_idx = np.argmax(similarities)
    print(f"Query: '{query}'")
    print(f"  Best match: '{docs[best_idx]}' (sim={similarities[best_idx]:.3f})")
    print()

print('This is how RAG retrieval works (Module 6)!')


---
## Exercise 6 (Challenge): Build a BPE Tokenizer From Scratch
Implement the byte-pair encoding algorithm step by step.

In [ ]:
from collections import Counter

def get_pairs(word_freqs):
    pairs = Counter()
    for word, freq in word_freqs.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    new_freqs = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in word_freqs.items():
        new_word = word.replace(bigram, replacement)
        new_freqs[new_word] = freq
    return new_freqs

# Training corpus
corpus = ['low', 'low', 'low', 'low', 'low',
          'lowest', 'lowest',
          'newer', 'newer', 'newer', 'newer', 'newer', 'newer',
          'wider', 'wider', 'wider']

# Initialize: split each word into characters
word_freqs = Counter()
for word in corpus:
    word_freqs[' '.join(list(word)) + ' </w>'] += 1

print('BPE TOKENIZER FROM SCRATCH')
print('=' * 50)
print(f'Initial vocab: {set(" ".join(word_freqs.keys()).split())}')

num_merges = 10
for i in range(num_merges):
    pairs = get_pairs(word_freqs)
    if not pairs: break
    best_pair = max(pairs, key=pairs.get)
    word_freqs = merge_pair(best_pair, word_freqs)
    print(f'  Merge {i+1}: {best_pair[0]} + {best_pair[1]} → {best_pair[0]+best_pair[1]} (freq={pairs[best_pair]})')

print(f'\nFinal vocabulary:')
vocab = set()
for word in word_freqs:
    vocab.update(word.split())
print(f'  {sorted(vocab)}')
print(f'\nBPE learns common subwords: "low", "er", "est" become single tokens!')


---
**6 exercises complete.** Tokenization concepts mastered: BPE, token tax, special tokens, contextual embeddings, semantic search, tokenizer training.